# Scoring what a model generates (images)

_Metrics & Evaluation — measuring models, data & statistics_

**When a model invents a picture, there is no single right answer — so we compare the whole pile of fakes to the pile of real ones.**

For a classifier you can check each prediction against a known label. For a generator there is no label — a newly invented face is not "right" or "wrong", it is just a new face. So you cannot score one image in isolation.

       The core trick: compare the whole distribution of generated samples to the whole distribution of real ones. "Distribution" just means the cloud of all your images viewed as points. If the cloud of fakes sits right on top of the cloud of reals — same center, same spread, same shape — the generator is good.

---

This notebook builds these generation metrics up one step at a time — first the from-scratch Frechet distance, then the library calls practitioners actually use. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — torchmetrics

There is no per-image label for a generated picture, so every metric here scores a *set* against another set. We build it in three steps: (1) the FID distance written from scratch, (2) the library calls for distribution metrics (FID, IS), (3) the reconstruction metrics that compare an output to a known target.

### Step 1 — Write the Frechet (FID) distance from scratch

FID treats each pile of images as a cloud of feature vectors and compares the two clouds with one formula: the squared gap between their means plus a trace term that compares their covariances. The matrix square root `sqrtm` can pick up a tiny imaginary part from numerical fuzz, so we drop it. Lower is better, and `0` means the two clouds match exactly.

In [ ]:
import numpy as np
from scipy.linalg import sqrtm


# ---------- from-scratch Frechet (FID) distance between two feature clouds ----------
def frechet_distance(feats_real, feats_fake):
    """FID = ||mu_r - mu_g||^2 + Tr(S_r + S_g - 2*(S_r S_g)^{1/2}).

    feats_*: arrays of shape (n_images, n_features), e.g. Inception activations.
    Lower is better; 0 means the two clouds match exactly.
    """
    mu_r = feats_real.mean(0)               # mean feature vector of the real cloud
    mu_g = feats_fake.mean(0)               # mean feature vector of the fake cloud
    S_r = np.cov(feats_real, rowvar=False)  # covariance of the real cloud
    S_g = np.cov(feats_fake, rowvar=False)  # covariance of the fake cloud

    diff = mu_r - mu_g                      # gap between the two centers
    covmean = sqrtm(S_r @ S_g)              # matrix square root of the product
    if np.iscomplexobj(covmean):            # numerical fuzz can add tiny imaginary parts
        covmean = covmean.real

    center_term = diff @ diff               # squared distance between means
    spread_term = np.trace(S_r + S_g - 2 * covmean)   # covariance mismatch
    return float(center_term + spread_term)

### Step 2 — Distribution metrics on a whole pile (FID, Inception Score)

These score a *generator*, not a single image, so they take batches of images. **FID** compares the fake cloud to the real cloud (lower is better). **Inception Score** rates sharpness and cross-class variety from the fakes alone (higher is better), so it never sees the real set. Both expect `uint8` tensors shaped `(N, 3, H, W)`.

In [ ]:
import torch
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

# images are uint8 tensors of shape (N, 3, H, W) for FID/IS
real = torch.randint(0, 255, (64, 3, 299, 299), dtype=torch.uint8)
fake = torch.randint(0, 255, (64, 3, 299, 299), dtype=torch.uint8)

# FID: quality + diversity of a generator (lower is better)
fid = FrechetInceptionDistance(feature=2048, normalize=False)
fid.update(real, real=True)
fid.update(fake, real=False)
print("FID:", fid.compute().item())

# Inception Score: sharpness + cross-class variety (higher is better; ignores real data)
iscore = InceptionScore()
iscore.update(fake)
mean_is, std_is = iscore.compute()
print("IS:", mean_is.item(), "+/-", std_is.item())

### Step 3 — Reconstruction metrics against a known target (SSIM, PSNR, LPIPS)

When you *do* have the right answer for each output — denoising, super-resolution — you compare output to target directly. **SSIM** rates local structure (1.0 = identical), **PSNR** is rescaled error in decibels (higher = closer), and **LPIPS** is a learned perceptual distance (lower = more similar). These take float tensors, and LPIPS expects inputs scaled into `[-1, 1]`.

In [ ]:
from torchmetrics.image import (
    StructuralSimilarityIndexMeasure,        # SSIM
    PeakSignalNoiseRatio,                     # PSNR
)
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

# --- reconstruction metrics: compare an output to its known target (float in [0,1]) ---
out    = torch.rand(8, 3, 128, 128)
target = torch.rand(8, 3, 128, 128)

ssim = StructuralSimilarityIndexMeasure(data_range=1.0)   # 1.0 = identical
print("SSIM:", ssim(out, target).item())

psnr = PeakSignalNoiseRatio(data_range=1.0)               # decibels; higher = closer
print("PSNR:", psnr(out, target).item())

# LPIPS: learned PERCEPTUAL distance (lower = more similar); inputs in [-1, 1]
lpips = LearnedPerceptualImagePatchSimilarity(net_type="alex", normalize=True)
print("LPIPS:", lpips(out, target).item())

# KID lives at torchmetrics.image.kid.KernelInceptionDistance (unbiased; better on small samples).
# CLIPScore lives at torchmetrics.multimodal.clip_score.CLIPScore (text-to-image match).

## Visualize the data & results

_Does a real FID-style distance stay tiny for two halves of the SAME class, and blow up for a different class or a noised 'generated' set?_

We stand in for an Inception network with PCA, then measure the Frechet distance in three scenarios to see the metric behave as expected.

### Step 1 — Build a feature space and three comparison sets

We load the 8x8 digits, scale the pixels into `[0, 1]`, and run PCA down to 10 dimensions — a cheap stand-in for the Inception feature space FID normally uses. Then we carve out three things to compare: two random halves of class 0 (should match), all of class 1 (a different class), and class 0 plus Gaussian noise (a fake 'generated' set).

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from scipy.linalg import sqrtm

d = load_digits()                          # 1797 real 8x8 handwritten digits
X = d.data / 16.0                          # scale pixels into [0, 1]
y = d.target

# PCA -> a shared 10-D 'feature space' standing in for an Inception network
F = PCA(n_components=10, random_state=0).fit_transform(X)
c0 = F[y == 0]                             # real class 0
c1 = F[y == 1]                             # real class 1

rng = np.random.RandomState(0)
idx = rng.permutation(len(c0))
half_a = c0[idx[:len(c0) // 2]]            # first half of class 0
half_b = c0[idx[len(c0) // 2:]]           # second half of class 0
noised = c0 + rng.normal(0, 1.0, c0.shape)   # a 'generated' set = class 0 + noise

### Step 2 — Score the three scenarios

We reuse the exact FID formula and apply it to each pair. Two halves of the same class should give a near-zero distance; a genuinely different class and the noised set should both blow up. The printed numbers confirm the metric rewards matching distributions and punishes mismatched ones.

In [ ]:
def frechet(P, Q):                         # the exact FID formula
    mu1 = P.mean(0)
    mu2 = Q.mean(0)
    s1 = np.cov(P, rowvar=False)
    s2 = np.cov(Q, rowvar=False)

    covmean = sqrtm(s1 @ s2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    diff = mu1 - mu2
    return float(diff @ diff + np.trace(s1 + s2 - 2 * covmean))

print("same class :", round(frechet(half_a, half_b), 2))   # 0.06
print("other class:", round(frechet(c0, c1), 2))           # 8.18
print("noised gen :", round(frechet(c0, noised), 2))       # 6.18

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your diffusion model's images look crisp and realistic, but a reviewer says "it only ever draws the same five faces." FID is mediocre. Which two-number diagnostic explains what is happening, and what would the numbers look like?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that FID mixes fidelity and diversity into one score, so it cannot tell you which is the problem. — _You need a metric that separates the two._
- Use precision (fidelity) and recall (diversity), or density and coverage. — _Precision asks "do fakes look real?", recall asks "do fakes cover the full real range?"._

**Answer:** Use precision & recall (or the more robust density & coverage). "Crisp and realistic" means the fakes land inside the real cloud, so precision (fidelity) is high. "Only five faces" means they cover just a sliver of the real range, so recall (diversity) is low. FID is mediocre precisely because it averages a great fidelity with a poor diversity into one middling number — these two metrics reveal the split.

</details>

**Problem 2.** You are denoising photos and have the clean target for each one. A teammate ranks methods by PSNR only and picks a model that produces visibly blurry results. Why did PSNR mislead, and which metrics should be added?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall PSNR is a function of mean squared error. — _Blur spreads small errors over many pixels, keeping MSE low._
- A sharp image with a tiny misalignment can have higher MSE than a blurry one. — _So PSNR can rank blur above sharpness, against human preference._

**Answer:** PSNR is just rescaled $\mathrm{MSE}$, and blur keeps MSE low while destroying detail, so PSNR rewards it. A sharp output that is off by a hair can score worse. Add SSIM / MS-SSIM (Structural / Multi-Scale Structural Similarity), which compare local structure, and LPIPS (Learned Perceptual Image Patch Similarity), which was trained to match human judgement. Reporting all three avoids the blur trap.

</details>

**Problem 3.** Real feature cloud: mean 5, variance 1. Generated cloud: mean 8, variance 1. Compute the FID (treat features as 1-D).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Center term: square the difference of means. — _$\lVert\mu_r-\mu_g\rVert^2=(5-8)^2=9$._
- Spread term: $\Sigma_r+\Sigma_g-2\sqrt{\Sigma_r\Sigma_g}=1+1-2\sqrt{1}=0$. — _Equal variances make the spread term vanish._

**Answer:** $\mathrm{FID}=9+0=\mathbf{9}$. The clouds have identical spread (variance 1 each, so the spread term is 0), and the whole distance comes from the centers being 3 apart: $(5-8)^2=9$. Move the generated mean to 5 and FID drops to 0.

</details>